# GNote Release Readiness Audit (Android + iOS)

- Date: 2026-03-13
- Scope: international production readiness for Play Store + App Store
- Inputs reviewed:
  - `android_manifest_patch.xml`
  - `ios_info_plist_patch.xml`
  - platform config files in `android/` and `ios/`

## Verdict
**NO-GO** for production release right now.

## Blocking Findings (Must Fix Before Release)

### 1) Android manifest patch not applied in actual manifest
- Current manifest lacks required permissions and notification receiver wiring from your patch guidance.
- This can break notifications and key runtime flows in production.

References:
- `android/app/src/main/AndroidManifest.xml:1`
- `android_manifest_patch.xml:22`
- `android_manifest_patch.xml:58`

### 2) iOS Info.plist patch not applied in actual Info.plist
- Missing `LSApplicationQueriesSchemes` entries for WhatsApp launch checks.
- This can cause `canLaunchUrl` behavior issues on device.

References:
- `ios/Runner/Info.plist:4`
- `ios_info_plist_patch.xml:13`

### 3) Default package/bundle IDs still in place (`com.example.gnote`)
- Not suitable for production publishing.
- Needs your owned/release namespace and aligned IDs across Android/iOS.

References:
- `android/app/build.gradle.kts:24`
- `android/app/src/main/kotlin/com/example/gnote/MainActivity.kt:1`
- `ios/Runner.xcodeproj/project.pbxproj:375`

### 4) International scale risk: timezone behavior is still region-hardcoded
- Core flows still assume `Africa/Harare` in multiple places.
- This can produce incorrect lock/reminder behavior for global users.

References:
- `lib/services/auth_service.dart:91`
- `lib/services/providers.dart:129`
- `lib/services/local_db.dart:24`

### 5) No localization/i18n implementation for international launch
- User-facing strings are centralized but English-only.
- International release should at least include i18n framework + fallback strategy.

Reference:
- `lib/core/constants.dart:220`

## Non-Blocking But Important
- Existing sync behavior still swallows many remote errors; app can appear synced when cloud writes failed.
- This is a reliability risk for production trust, though not a store-submission blocker by itself.

## Minimum Go-Live Checklist
1. Apply Android manifest patch items into real `AndroidManifest.xml`.
2. Apply iOS `Info.plist` patch keys (`LSApplicationQueriesSchemes`, notification usage text as needed).
3. Replace `com.example.gnote` IDs with production IDs on both platforms.
4. Decide and implement global timezone strategy (device timezone or profile timezone consistently).
5. Add localization scaffolding and at least primary launch languages.
6. Run release smoke tests on physical Android + iPhone for:
   - auth signup/login/OTP
   - notifications (permission + schedule + tap routing)
   - WhatsApp launch flow
   - offline/online sync transitions

## Final Recommendation
Do not submit yet. Resolve the blockers above first, then do a final release-candidate pass.